In [0]:
# Objetive: To undestand the structure, quality and quirks of the two files before solving the exercises

# Files:
# orders.csv --> factual information regarding the orders received
# invoicing_data.json --> invoices for each order

import json
import re
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder.getOrCreate()


In [0]:
# After manual inspection of the csv --> headers, separator and quote/escape characters identified  
df_orders_raw = spark.read \
    .option("header", "true") \
    .option("sep", ";") \
    .option("quote", '"') \
    .option("escape", '"') \
    .csv(ORDERS_PATH)

df_orders_raw.printSchema()

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8074509237248022>, line 7
      1 # After manual inspection of the csv --> headers, separator and quote/escape characters identified  
      2 df_orders_raw = spark.read \
      3     .option("header", "true") \
      4     .option("sep", ";") \
      5     .option("quote", '"') \
      6     .option("escape", '"') \
----> 7     .csv(ORDERS_PATH)
      9 df_orders_raw.printSchema()

NameError: name 'ORDERS_PATH' is not defined

Discovery --> date as string

In [0]:

# After manual inspection of the json --> multiline
df_invoices_raw = spark.read \
    .option("multiline", "true") \
    .json(INVOICING_PATH)

# Aplanar
df_invoices = df_invoices_raw \
    .select(F.explode("data.invoices").alias("inv")) \
    .select("inv.*")

df_invoices.printSchema()

Discovery
--> grossValue, vat as strings

In [0]:
# Counts
n_orders    = df_orders_raw.count()
n_invoices  = df_invoices.count()
n_cols_ord  = len(df_orders_raw.columns)
n_cols_inv  = len(df_invoices.columns)
 
print(f"\n[orders]    {n_orders} filas  × {n_cols_ord} columnas")
print(f"[invoices] {n_invoices} filas  × {n_cols_inv} columnas")

Discovery --> few invoices (~18% orders)

In [0]:
# Nulls & empties
def null_report(df, label):
    print(f"\n[{label}] Nulos y cadenas vacías:")
    exprs = [
        F.sum(
            (F.col(c).isNull() | (F.trim(F.col(c)) == "")).cast("int")
        ).alias(c)
        for c in df.columns
    ]
    df.select(exprs).show(1, truncate=False)
 
null_report(df_orders_raw, "orders")
# HALLAZGOS esperados:
#   - contact_data: ~8 filas vacías/nulas  → placeholder "John Doe" / "Unknown"
#   - salesowners:  0 nulos (siempre hay al menos uno)
 
null_report(df_invoices, "invoicing")
# HALLAZGOS esperados: sin nulos en los campos clave

Discovery
- Only nulls found in contact data, placeholder might be needed

In [0]:
# Duplicated order_id
print("\n[orders] duplicated IDs (order_id):")
dup_orders = df_orders_raw.groupBy("order_id") \
    .count() \
    .filter(F.col("count") > 1)
dup_orders.show(truncate=False)
 
 # Duplicated invoice_id
print("\n[invoicing] duplicated IDs (invoice_id):")
dup_inv_id = df_invoices.groupBy("id") \
    .count() \
    .filter(F.col("count") > 1)
dup_inv_id.show(truncate=False)
 
 # Duplicated order_id in invoices
print("\n[invoicing] Same order_id in multiple invoices:")
dup_order_inv = df_invoices.groupBy("orderId") \
    .count() \
    .filter(F.col("count") > 1)
dup_order_inv.show(truncate=False)


Discoveries 
- Orders do not have duplicated IDs (order_id)
- Invoices do not have duplicated IDs (invoice_id)
- Invoices have one duplicated order_id

In [0]:
# We have to investigate if the duplicated order_ID comes from a duplicated line or it is a different issue
duplicated_ids = df_invoices.groupBy("order_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .select("order_id")

# Rows with duplicated order_id
df_only_duplicates = df_orders_raw.join(duplicated_ids, on="order_id", how="inner")

# Compare with distinct
total_rows = df_only_duplicates.count()
distinct_rows = df_only_duplicates.distinct().count()

print("Result:")
if total_rows > distinct_rows and distinct_rows == duplicated_ids.count():
    print("Exact duplicates")
else:
    print("Not duplicates")


In [0]:
# Format of the date field
 
df_orders_raw.select("date").distinct().orderBy("date").show(20)
# HALLAZGO: formato DD.MM.YY (ej. "29.01.22") → se parseará con
# to_date(date, 'dd.MM.yy').  Rango de datos: 2021-2025.